In [0]:
-- Recria a tabela de pedidos, agora com mais pedidos de produtos baratos (substitui a anterior)
CREATE OR REPLACE TABLE ai_bi_data_analyst.ai_bi_dashboards.ca_orders AS
WITH
-- Lista com todos os IDs de clientes (1 linha com um array)
cust AS (
  SELECT collect_list(customerid) AS ids
  FROM ai_bi_data_analyst.ai_bi_dashboards.ca_customers
),
-- Gera 15.536 pedidos com valores aleatórios (sementes fixas deixam o resultado reproduzível)
base AS (
  SELECT
    CAST(id AS INT) AS orderid,                                                              -- número do pedido: 1 a 15536
    element_at(cust.ids, 1 + CAST(FLOOR(rand(2) * size(cust.ids)) AS INT)) AS customerid,   -- cliente sorteado de ca_customers
    rand(3) AS r_prod,                                                                      -- número aleatório usado para escolher o produto
    DATE_ADD(DATE'2021-11-16', CAST(FLOOR(rand(4) * 1142) AS INT)) AS orderdate,            -- data entre 2021-11-16 e 2024-12-31
    300 + CAST(FLOOR(rand(5) * 251) AS INT) AS quantity,                                    -- quantidade entre 300 e 550
    element_at(array('Hannah','Shalini','Ethan','Sam','Ingrid'),
               1 + CAST(FLOOR(rand(6) * 5) AS INT)) AS salesrep                             -- vendedor sorteado
  FROM range(1, 15537)
  CROSS JOIN cust
),
-- Escolhe o produto com pesos: os baratos aparecem muito mais que os caros
picked AS (
  SELECT
    orderid, customerid, orderdate, quantity, salesrep,
    CASE
      WHEN r_prod < 0.26  THEN 'A802a'    -- 26% Mill
      WHEN r_prod < 0.43  THEN 'JQ9322'   -- 17% TableSaw
      WHEN r_prod < 0.59  THEN 'R9S10'    -- 16% Lathe
      WHEN r_prod < 0.74  THEN 'W8931'    -- 15% Planer
      WHEN r_prod < 0.88  THEN 'BC111d'   -- 14% Drill
      WHEN r_prod < 0.96  THEN 'OR307p'   --  8% Kazoo
      WHEN r_prod < 0.98  THEN 'Q28R3'    --  2% Grinder
      WHEN r_prod < 0.995 THEN 'EE05x'    -- 1,5% Press
      ELSE 'MDb304'                       -- 0,5% CAD System
    END AS productid
  FROM base
)
SELECT
  pk.orderid,
  pk.customerid,
  pk.productid,
  pk.orderdate,
  pk.quantity,
  CAST(pk.quantity * p.listprice AS DECIMAL(12,2)) AS orderamt,  -- valor = quantidade x preço de lista
  pk.salesrep
FROM picked pk
JOIN ai_bi_data_analyst.ai_bi_dashboards.ca_products p
  ON pk.productid = p.productid;

-- Confere: total vendido por ano (compare com as metas: 9M, 80M, 87M, 86M)
SELECT YEAR(orderdate) AS year, ROUND(SUM(orderamt), 2) AS total_vendido
FROM ai_bi_data_analyst.ai_bi_dashboards.ca_orders
GROUP BY YEAR(orderdate)
ORDER BY year;